# FactorFolio Known-Answer Tests

**AC420 project, Jasper Lin.** Before the app uses the code in `core/`, this notebook checks that it reproduces answers from class. Test numbers match the known-answer tests in my Build Guide.

* **Tests 1 and 2:** the Lecture 12 & 13 notebook's CAPM and Fama-French regressions for MSFT and XOM (2016 to 2020).
* **Test 3:** the Lecture 17 study guide's MSFT result after the year-month merge fix.
* **Portfolio check:** the portfolio's beta equals the weighted average of its stocks' betas, and R-squared equals the systematic share of risk.
* **Tests 4 and 5:** the DCF in `core/valuation.py`: the Lecture 15 Coca-Cola example and the Lecture 16 study guide's drill 2.
* **Test 6:** peer comparables: the Lecture 16 Amazon example (mean and median peer P/E) and drill 3's 5% band.
* **Test 8:** the Altman Z-score in `core/models.py`: the Lecture 10 General Motors example and test drill Q1.
* **Test 9:** the Sharpe ratio in `core/models.py`: the Lecture 18 study guide's worked example and drills.
* **Test 10:** the range for beta (professor's check-in feedback): the 95% confidence interval from the regression and the worst, most likely and best case in `core/projections.py`.
* **Test 7:** the bear, base and bull projections in `core/projections.py` (Build Guide known-answer test 7, from outline section 5.5). This one needs no class files.

**To run in Colab:** upload `Lect-12-output.csv` and `FF-Factors-2016-2020.csv` (Lecture 12 folder) and `stock_returns.csv` and `FF_factors.csv` (Lecture 17 folder), then Runtime, then Run all. The class data stays in Colab; it is never added to the GitHub repo.

Written with AI assistance (Claude); see my AI usage log.

### Get the app's code from GitHub and import the models

In [ ]:
!git clone -q https://github.com/jasperl12/ac420-project-factorfolio-portfolio-risk-app.git factorfolio
import sys
sys.path.append('factorfolio')

import pandas as pd
from core import models

DATA_DIR = '/content/'

### Tests 1 and 2: Lecture 12 & 13 notebook (MSFT and XOM, 60 months)
Same steps as the class notebook: split the stacked file by ticker, rename `dateff` to `date`, merge, and build the excess return `RET - rf`.

In [ ]:
data = pd.read_csv(DATA_DIR + 'Lect-12-output.csv')
ff_factors = pd.read_csv(DATA_DIR + 'FF-Factors-2016-2020.csv')
ff_factors.rename(columns={'dateff': 'date'}, inplace=True)

results = []
for tic in ['MSFT', 'XOM']:
    stock_ff = pd.merge(data[data['TICKER'] == tic], ff_factors, on='date', how='inner')
    print(tic, 'rows after merge:', len(stock_ff))       # should be 60
    y = stock_ff['RET'] - stock_ff['rf']
    capm = models.run_capm(y, stock_ff['mktrf'])
    ff3 = models.run_ff3(y, stock_ff)
    results.append({'ticker': tic, 'capm_alpha': capm['alpha'], 'capm_beta': capm['beta'], 'capm_r2': capm['r2'],
                    'ff3_alpha': ff3['alpha'], 'ff3_beta': ff3['beta'], 'ff3_s': ff3['s'], 'ff3_h': ff3['h'],
                    'ff3_r2': ff3['r2']})
lecture12 = pd.DataFrame(results).set_index('ticker')
lecture12.round(4)

In [ ]:
# Answers from the Lecture 12 and 14 study guides
expected = pd.DataFrame({
    'capm_alpha': [0.0157, -0.0211], 'capm_beta': [0.756, 1.293], 'capm_r2': [0.452, 0.616],
    'ff3_alpha': [0.0114, -0.0149], 'ff3_beta': [0.938, 1.092], 'ff3_s': [-0.498, 0.429],
    'ff3_h': [-0.349, 0.554], 'ff3_r2': [0.560, 0.695]}, index=['MSFT', 'XOM'])
gap = (lecture12 - expected).abs()
print(gap.round(4))
assert (gap < 0.001).all().all(), 'A Lecture 12 answer does not match'
print('Tests 1 and 2 passed: every estimate matches the class answers to 3 decimals.')

### Test 3: Lecture 17 study guide (MSFT after the year-month merge fix)
Expected: 64 months, CAPM beta 1.064, FF3 R-squared 0.614.

In [ ]:
factors = pd.read_csv(DATA_DIR + 'FF_factors.csv').rename(columns={'dateff': 'datadate'})
returns = pd.read_csv(DATA_DIR + 'stock_returns.csv', usecols=['datadate', 'tic', 'trt1m'])
returns = returns.dropna(subset=['trt1m'])
returns['ret'] = returns['trt1m'] / 100
returns['ym'] = pd.to_datetime(returns['datadate']).dt.to_period('M')
factors['ym'] = pd.to_datetime(factors['datadate']).dt.to_period('M')
combined = pd.merge(returns, factors.drop(columns='datadate'), on='ym', how='inner')

msft = combined[combined['tic'] == 'MSFT']
y = msft['ret'] - msft['rf']
capm = models.run_capm(y, msft['mktrf'])
ff3 = models.run_ff3(y, msft)
print('months:', capm['n_months'], '| CAPM beta:', round(capm['beta'], 3), '| FF3 R2:', round(ff3['r2'], 3))
print(models.alpha_label(capm))
assert capm['n_months'] == 64 and abs(capm['beta'] - 1.064) < 0.001 and abs(ff3['r2'] - 0.614) < 0.001
print('Test 3 passed.')

### Portfolio check (sample portfolio from the app)
* The portfolio's monthly return is the weighted sum of its stocks' returns.
* Over the same months, the portfolio's beta, size and value loadings equal the weighted averages of its stocks' loadings (the suggestions engine relies on this).
* R-squared equals the systematic share of risk, and diversification should make the portfolio's R-squared higher than the typical stock's.

In [ ]:
weights = {'AAPL': 30, 'MSFT': 20, 'NVDA': 20, 'KO': 15, 'JNJ': 15}
wide = combined.pivot_table(index='ym', columns='tic', values='ret')[list(weights)]
window = wide.join(factors.set_index('ym')[['mktrf', 'smb', 'hml', 'rf']], how='inner').dropna()
ds = {'window': window, 'tickers': list(weights)}

result = models.analyze_portfolio(ds, weights, benchmark=None)
stocks, port = result['stocks'], result['portfolio']
print(stocks[['weight', 'beta', 's', 'h', 'r2', 'volatility']].round(3))

w = result['weights']
checks = {'beta': (stocks['beta'] * w).sum(), 'ff3_beta': (stocks['ff3_beta'] * w).sum(),
          's': (stocks['s'] * w).sum(), 'h': (stocks['h'] * w).sum()}
for k, weighted_avg in checks.items():
    print(f'portfolio {k}: {port[k]:.4f}  weighted average of stocks: {weighted_avg:.4f}')
    assert abs(port[k] - weighted_avg) < 1e-10
assert abs(port['split']['systematic_share'] - port['r2']) < 1e-10
print('portfolio R2:', round(port['r2'], 3), '| average stock R2:', round(stocks['r2'].mean(), 3))
assert port['r2'] > stocks['r2'].mean()
print('Portfolio check passed.')

### Test 4: DCF, Lecture 15 notebook (Coca-Cola)
Inputs from the class notebook: EPS 3.30511 (0y) and 3.52666 (+1y), beta 0.318, Rf 5.3%, ERP 5.7%, long-term growth 3%.
Expected: cost of equity 7.11% and a value of $85.91.

In [ ]:
from core import valuation

re = valuation.cost_of_equity(0.053, 0.318, 0.057)
price1 = valuation.dcf_price(3.30511, 3.52666, re, g_lt=0.03)
print('cost of equity:', round(re, 4), '| predicted price:', round(price1, 2))
assert round(re, 4) == 0.0711 and round(price1, 2) == 85.91
print('Test 4 passed.')

### Test 5: DCF, Lecture 16 study guide drill 2
EPS 4.00 and 4.40, cost of equity 10%, long-term growth 3%. Expected: $64.42.
Also checks the hygiene rules: a negative EPS or a cost of equity at or below long-term growth leaves the stock out.

In [ ]:
price1 = valuation.dcf_price(4.00, 4.40, 0.10, g_lt=0.03)
print('predicted price:', round(price1, 2))
assert round(price1, 2) == 64.42
print(valuation.dcf_problems(-1.0, 4.40, 0.10, 0.03))     # EPS at or below zero
print(valuation.dcf_problems(4.00, 4.40, 0.03, 0.03))     # re not above long-term growth
print(valuation.dcf_problems(float('nan'), 4.40, 0.10, 0.03))
assert valuation.dcf_problems(4.00, 4.40, 0.10, 0.03) == []
print('Test 5 passed.')

### Test 6: Peer comparables, Lecture 16 (Amazon) and drill 3
Peer forward P/Es 10.87 (BABA), 6.26 (PDD) and 29.90 (MELI); Amazon EPS $12.91.
Expected: mean P/E 15.68 gives $202.45 and the median P/E 10.87 gives about $140.32 (the class used P/Es before rounding, so allow a few cents).
Drill 3: peer P/Es 8, 14 and 35, EPS $4.00, price $62: the mean says underpriced, the median says overpriced.

In [ ]:
amzn = valuation.comps_price([10.87, 6.26, 29.90], 12.91)
print(amzn)
assert abs(amzn['price_mean'] - 202.45) < 0.10 and abs(amzn['price_median'] - 140.32) < 0.05

# Bad peers are dropped before averaging (checks 19 and 20)
messy = valuation.comps_price([10.87, None, -5.0, 6.26, 29.90], 12.91)
assert messy['n_peers'] == 3 and messy['price_median'] == amzn['price_median']

drill = valuation.comps_price([8, 14, 35], 4.00)
mean_call = valuation.price_label(62, drill['price_mean'], drill['price_mean'])
median_call = valuation.price_label(62, drill['price_median'], drill['price_median'])
print('mean P/E', drill['mean_pe'], '->', mean_call, '| median P/E', drill['median_pe'], '->', median_call)
assert (mean_call, median_call) == ('looks underpriced', 'looks overpriced')
print('Test 6 passed.')

### Test 7: Bear, base and bull projections (Build Guide known-answer test 7)
Outline section 5.5 example: Rf 4.5%, ERP 5.7%, beta 1.2, $10,000 for 5 years, base case.
* Portfolio return = 4.5% + 1.2 x 5.7% = 11.34% a year, so $10,000 grows to about $17,100.
* SPY return = 4.5% + 5.7% = 10.2% a year, so $10,000 grows to about $16,250.

In [ ]:
from core import projections

base = projections.project(10000, 5, beta=1.2, rf=0.045, scenario='Base')
print(base['path'].round(0))
print('portfolio return:', round(base['portfolio_return_first'], 4), '| SPY return:', round(base['spy_return_first'], 4))
assert abs(base['portfolio_return_first'] - 0.1134) < 1e-10      # 4.5% + 1.2 x 5.7%
assert abs(base['spy_return_first'] - 0.102) < 1e-10             # 4.5% + 5.7%
assert abs(base['portfolio_value'] - 10000 * 1.1134 ** 5) < 0.01  # future value, compounded 5 times
assert round(base['portfolio_value'], -2) == 17100 and round(base['spy_value'], -1) == 16250
print('Base case matches the outline example.')

In [ ]:
# Bear and bull: the scenario return in year 1, then the base return in years 2 to 5
scen = projections.load_scenarios()
for name, key in [('Bear', 'bear_market_return'), ('Bull', 'bull_market_return')]:
    p = projections.project(10000, 5, beta=1.2, rf=0.045, scenario=name)
    year1 = 0.045 + 1.2 * (scen[key] - 0.045)                   # CAPM prediction form
    assert abs(p['portfolio_return_first'] - year1) < 1e-12
    assert abs(p['portfolio_return_later'] - 0.1134) < 1e-12    # back to base after year 1
    assert abs(p['spy_return_first'] - scen[key]) < 1e-12       # SPY earns the market return
    print(f"{name}: market {scen[key]:.2%}, portfolio year 1 {year1:.2%}, "
          f"after 5 years ${p['portfolio_value']:,.0f} vs. SPY ${p['spy_value']:,.0f}")

# A portfolio with beta 1 should track SPY exactly in every scenario
for name in projections.SCENARIOS:
    assert abs(projections.project(10000, 5, beta=1.0, rf=0.045, scenario=name)['difference']) < 1e-8
print(projections.assumptions_table(base))
print('Test 7 passed.')

### Test 8: Altman Z-score, Lecture 10 (General Motors, 2007)
GM's ratios from the Lecture 10 study guide: T1 -0.067, T2 -0.262, T3 -0.030, T4 0.038, T5 1.21. Expected: Z about 0.67, distress zone.
Lecture 10 drill Q1: WC/TA 0.10, RE/TA 0.15, EBIT/TA 0.08, ME/TL 0.90, S/TA 1.10. Expected: Z = 2.22, grey zone.
The last part builds GM's ratios from the dollar amounts in the study guide, the same way the app builds them from Yahoo's statements.

In [ ]:
gm = models.altman_z(-0.067, -0.262, -0.030, 0.038, 1.21)
print('GM Z:', round(gm, 2), models.altman_zone(gm))
assert round(gm, 2) == 0.67 and models.altman_zone(gm) == 'distress'

q1 = models.altman_z(0.10, 0.15, 0.08, 0.90, 1.10)
print('Q1 Z:', round(q1, 2), models.altman_zone(q1))
assert round(q1, 2) == 2.22 and models.altman_zone(q1) == 'grey'

# GM from dollar amounts ($ billions): working capital 60 - 70, retained earnings -39, EBIT -4.4,
# total assets 149, market value of equity 7 (here 1 share x $7), total liabilities 186, sales 181
items = {'current_assets': 60, 'current_liabilities': 70, 'total_assets': 149, 'retained_earnings': -39,
         'total_liabilities': 186, 'shares': 1, 'ebit': -4.4, 'revenue': 181}
gm_row = models.altman_for_stock(items, price=7, sector='Consumer Cyclical', industry='Auto Manufacturers')
print({k: round(gm_row[k], 3) for k in ['T1', 'T2', 'T3', 'T4', 'T5', 'z']}, gm_row['zone'])
assert gm_row['zone'] == 'distress' and abs(gm_row['z'] - 0.68) < 0.01   # 0.68 before rounding the ratios

# Banks and insurers are left out (check 15)
print(models.altman_for_stock(items, 7, 'Financial Services', 'Banks - Diversified')['status'])
print('Test 8 passed.')

### Test 9: Sharpe ratio, Lecture 18
Study guide worked example: mean monthly return 1.20%, rf 0.25% a month, standard deviation 5.00%. Expected Sharpe = 0.19 (monthly).
Drill Q1: mean 1.5%, rf 0.25%, standard deviation 6%. Expected 0.208. Annualizing the notebook's AAPL result: 0.168 x sqrt(12) is about 0.58.
The app's function is checked on a made-up return series built to have exactly those means and standard deviations, and against the Assignment #5 notebook's shortcut (dividing by the raw returns' standard deviation), which gives the same answer when rf is constant (drill Q2).

In [ ]:
import numpy as np

def series_with(mean, std, n=60):
    z = np.random.default_rng(0).normal(size=n)
    z = (z - z.mean()) / z.std(ddof=1)            # exactly mean 0 and sample std 1
    return pd.Series(mean + std * z)

rf = pd.Series([0.0025] * 60)
worked = models.sharpe_ratio(series_with(0.012, 0.05), rf)
drill = models.sharpe_ratio(series_with(0.015, 0.06), rf)
print('worked example:', round(worked['monthly'], 3), '| drill Q1:', round(drill['monthly'], 3))
assert round(worked['monthly'], 2) == 0.19 and round(drill['monthly'], 3) == 0.208

# Drill Q2: with a constant rf, std(R - rf) = std(R), so the notebook's shortcut gives the same Sharpe ratio
R = series_with(0.015, 0.06)
notebook = (R - 0.0025).mean() / R.std()
assert abs(notebook - drill['monthly']) < 1e-12
print('annualized AAPL example:', round(0.168 * 12 ** 0.5, 2))
assert round(0.168 * 12 ** 0.5, 2) == 0.58
print('Test 9 passed.')

### Test 10: range for beta, worst, most likely and best case
Beta is an estimate, so the regression's 95% confidence interval (the `[0.025  0.975]` columns in `model.summary()`) gives a low and a high beta.
Part A checks the interval on the Lecture 12 MSFT regression: it must equal beta ± t × standard error, with t from the t-distribution with n - 2 degrees of freedom.
Part B checks the 3 x 3 table by hand: $10,000 for 5 years, Rf 4.5%, beta 1.2 with a range of 1.0 to 1.4. Most likely = base market with beta 1.2 = 10,000 × 1.1134^5 = $17,110 (the same answer as test 7).
Worst = bear market with the low beta (only year 1 is a bear year, and the base years after it reward a higher beta) = $14,771; best = bull market with the high beta = $21,733.
With a 1-year horizon the high beta becomes the worst case in a bear market: 10,000 × (1 + 0.045 + 1.4 × (0.00155 - 0.045)) = $9,842.

In [ ]:
from scipy import stats
from core import projections

# Part A: the confidence interval is beta +/- t x standard error
msft = pd.merge(data[data['TICKER'] == 'MSFT'], ff_factors, on='date', how='inner')
capm = models.run_capm(msft['RET'] - msft['rf'], msft['mktrf'])
t = stats.t.ppf(0.975, capm['model'].df_resid)
low, high = capm['beta_ci']
print(f"MSFT beta {capm['beta']:.3f}, standard error {capm['beta_se']:.3f}, 95% interval {low:.3f} to {high:.3f}")
assert abs(low - (capm['beta'] - t * capm['beta_se'])) < 1e-9 and abs(high - (capm['beta'] + t * capm['beta_se'])) < 1e-9

# Part B: the worst, most likely and best case, checked by hand
r = projections.beta_range(10000, 5, 1.2, 1.0, 1.4, 0.045)
print(r['grid'].round(0))
bear = projections.market_returns(0.045)['Bear']
assert round(r['likely']['value']) == round(10000 * 1.1134 ** 5) == 17110
assert (r['worst']['row'], r['worst']['scenario']) == ('Low beta', 'Bear')
assert round(r['worst']['value']) == round(10000 * (1 + 0.045 + 1.0 * (bear - 0.045)) * 1.102 ** 4) == 14771
assert (r['best']['row'], r['best']['scenario'], round(r['best']['value'])) == ('High beta', 'Bull', 21733)
one_year = projections.beta_range(10000, 1, 1.2, 1.0, 1.4, 0.045)
assert (one_year['worst']['row'], one_year['worst']['scenario']) == ('High beta', 'Bear')
assert round(one_year['worst']['value']) == round(10000 * (1 + 0.045 + 1.4 * (bear - 0.045))) == 9842
print('Test 10 passed.')